# BirdCLEF 2026 — Inference & Submission Notebook

A **Kaggle Competition Notebook** for BirdCLEF inference and submission generation.

**Data paths**
- Competition data: `/kaggle/input/competitions/birdclef-2026/`
- Models & thresholds: `/kaggle/input/datasets/sahilpoply/birdclef111/`

**Pipeline**
- preprocessing → TTA → ensemble → thresholds → `submission.csv`


## 1. Overview

This notebook follows a standard BirdCLEF inference flow:

1. Load `sample_submission.csv`
2. Discover test soundscapes
3. Window each soundscape into fixed-duration segments
4. Extract features per window (e.g., log-mel)
5. Run inference (optional TTA)
6. Ensemble (optional)
7. Apply thresholds (optional)
8. Write `submission.csv`

Notes:
- **Inference-only**: no training loops.
- Designed to be **public-notebook friendly** (clean sections, safe defaults).


## 2. Imports


In [ ]:
from __future__ import annotations

# =====================
# Standard libraries
# =====================
import gc
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Suppress noisy warnings (keep Kaggle logs clean)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# =====================
# ML / data libraries
# =====================
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

# (Used across the local pipeline scripts)
try:
    from torch.utils.data import DataLoader  # noqa: F401
except Exception:
    DataLoader = None  # type: ignore

try:
    from tqdm import tqdm  # noqa: F401
except Exception:
    tqdm = None  # type: ignore

try:
    import timm  # noqa: F401
except Exception:
    timm = None  # type: ignore

# Audio (local pipeline uses librosa)
try:
    import librosa  # type: ignore
except Exception:
    librosa = None

# Optional: used in many Kaggle notebooks, but not required
try:
    import soundfile as sf  # type: ignore
except Exception:
    sf = None

# =====================
# Device initialization
# =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device(DEVICE)

torch.set_float32_matmul_precision("high")
print("DEVICE:", DEVICE)


## 3. Config

Centralized paths and inference settings.

**Kaggle Competition paths**
- `COMPETITION_DIR` → `/kaggle/input/competitions/birdclef-2026`
- `TEST_DIR`, `SAMPLE_SUBMISSION`, `TRAIN_CSV`, `TAXONOMY_CSV`

**Kaggle Dataset paths (models & thresholds)**
- `MODEL_DATASET_DIR` → `/kaggle/input/datasets/sahilpoply/birdclef111`


In [ ]:
# Reused verbatim from `final_pipeline/configs/config.py` (optimized project defaults)
SR = 32000
DURATION = 5.0

N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 128

FMIN = 20
FMAX = 16000

EXPECTED_TIME_FRAMES = 313

MODEL_NAME = "tf_efficientnet_b0"
NUM_CLASSES = 206
DROPOUT = 0.3

PROJECT_DEVICE = "cpu"

# ============================================================
# Kaggle Competition paths (BirdCLEF 2026)
# ============================================================
COMPETITION_DIR = "/kaggle/input/competitions/birdclef-2026"

TEST_DIR = f"{COMPETITION_DIR}/test_soundscapes"
SAMPLE_SUBMISSION = f"{COMPETITION_DIR}/sample_submission.csv"
TRAIN_CSV = f"{COMPETITION_DIR}/train.csv"
TAXONOMY_CSV = f"{COMPETITION_DIR}/taxonomy.csv"

# ============================================================
# Kaggle Dataset paths (models & thresholds)
# ============================================================
MODEL_DATASET_DIR = "/kaggle/input/datasets/sahilpoply/birdclef111"

THRESHOLDS_CSV = f"{MODEL_DATASET_DIR}/optimized_thresholds.csv"
SOUNDSCAPE_MODEL_PATH = f"{MODEL_DATASET_DIR}/best_soundscape_model.pth"
PSEUDO_MODEL_PATH = f"{MODEL_DATASET_DIR}/best_pseudo_model.pth"

print("COMPETITION_DIR:", COMPETITION_DIR)
print("TEST_DIR:", TEST_DIR)
print("SAMPLE_SUBMISSION:", SAMPLE_SUBMISSION)
print("TRAIN_CSV:", TRAIN_CSV)
print("TAXONOMY_CSV:", TAXONOMY_CSV)
print("MODEL_DATASET_DIR:", MODEL_DATASET_DIR)
print("THRESHOLDS_CSV:", THRESHOLDS_CSV)
print("SOUNDSCAPE_MODEL_PATH:", SOUNDSCAPE_MODEL_PATH)
print("PSEUDO_MODEL_PATH:", PSEUDO_MODEL_PATH)
print("RUNTIME DEVICE:", DEVICE)


In [ ]:
# Path validation (run before inference)
print("Competition files:")
print(os.listdir(COMPETITION_DIR))

print("Test files:", len(os.listdir(TEST_DIR)))

if not os.path.exists(SAMPLE_SUBMISSION):
    raise FileNotFoundError(f"sample_submission.csv not found: {SAMPLE_SUBMISSION}")

if not os.path.exists(TRAIN_CSV):
    raise FileNotFoundError(f"train.csv not found: {TRAIN_CSV}")

if not os.path.exists(TAXONOMY_CSV):
    raise FileNotFoundError(f"taxonomy.csv not found: {TAXONOMY_CSV}")

print("✅ Competition paths validated")

# Model asset validation (before loading)
print("\nModel dataset files:")
print(os.listdir(MODEL_DATASET_DIR))

print("Threshold file path:", THRESHOLDS_CSV)
print("Soundscape model path:", SOUNDSCAPE_MODEL_PATH)
print("Pseudo model path:", PSEUDO_MODEL_PATH)

assert os.path.exists(THRESHOLDS_CSV)
assert os.path.exists(SOUNDSCAPE_MODEL_PATH)
assert os.path.exists(PSEUDO_MODEL_PATH)

print("✅ Model assets validated")


## 4. Model Definition

EfficientNet-B0 + AttentionPooling (copied from local project).

Checkpoints loaded from:
- `SOUNDSCAPE_MODEL_PATH`
- `PSEUDO_MODEL_PATH`


In [ ]:
# Inference-only model definition (copied from local project).
# Architecture: EfficientNet-B0 (`timm`) + AttentionPooling + Dropout + Linear head

class AttentionPooling(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [B, T, C]
        attention_weights = self.attention(x)
        attention_weights = torch.softmax(attention_weights, dim=1)
        weighted = x * attention_weights
        pooled = weighted.sum(dim=1)
        return pooled


class BirdCLEFModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Backbone (same as `src/models/efficientnet_model.py`)
        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=True,
            in_chans=3,
            num_classes=0,
        )

        feature_dim = self.backbone.num_features

        # Attention pooling (same as `src/models/attention.py`)
        self.attention_pool = AttentionPooling(feature_dim)

        # Dropout (same project value)
        self.dropout = nn.Dropout(DROPOUT)

        # Classifier head (same project value)
        self.classifier = nn.Linear(feature_dim, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Features: [B, C, H, W]
        features = self.backbone.forward_features(x)

        # Temporal pool over frequency -> [B, C, T]
        features = features.mean(dim=2)

        # [B, C, T] -> [B, T, C]
        features = features.permute(0, 2, 1)

        pooled = self.attention_pool(features)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits


def load_model(checkpoint_path: str) -> nn.Module:
    """Load model checkpoint from full Kaggle dataset path."""
    model = BirdCLEFModel()

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    state = torch.load(checkpoint_path, map_location=DEVICE)

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    model.load_state_dict(state, strict=True)
    model.to(TORCH_DEVICE)
    model.eval()

    print(f"✅ Loaded: {checkpoint_path}")
    return model


## 5. Audio Processing

In [ ]:
def read_audio(path: Path, sr: int) -> Tuple[np.ndarray, int]:
    """Read audio as mono float32.

    Uses `soundfile` or `librosa` if available.
    Fallback: returns zeros so the notebook stays runnable.
    """
    try:
        if sf is not None:
            audio, file_sr = sf.read(str(path), always_2d=False)
            if isinstance(audio, np.ndarray) and audio.ndim > 1:
                audio = np.mean(audio, axis=1)
            audio = audio.astype(np.float32)
            file_sr = int(file_sr)
            if file_sr != sr and librosa is not None:
                audio = librosa.resample(audio, orig_sr=file_sr, target_sr=sr)
                file_sr = sr
            return audio, file_sr
        if librosa is not None:
            audio, file_sr = librosa.load(str(path), sr=sr, mono=True)
            return audio.astype(np.float32), int(file_sr)
    except Exception as e:
        print(f"Warning: audio read failed for {path}: {e}")

    n = int(sr * DURATION)
    return np.zeros((n,), dtype=np.float32), sr


# --- Copied from `src/utils/audio_utils.py` (project preprocessing) ---

def create_mel_spectrogram(audio: np.ndarray) -> np.ndarray:
    """Project mel-spectrogram pipeline (log-mel in dB)."""
    if librosa is None:
        raise RuntimeError("librosa is required for mel spectrogram generation")

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


def normalize(x: np.ndarray) -> np.ndarray:
    """Project normalization: scale to [-1, 1]."""
    x_min = x.min()
    x_max = x.max()
    x = 2 * (x - x_min) / (x_max - x_min + 1e-8) - 1
    return x


def create_3_channel_tensor(mel_db: np.ndarray) -> np.ndarray:
    """Project 3-channel tensor: [mel, delta, delta2] all normalized."""
    if librosa is None:
        raise RuntimeError("librosa is required for delta features")

    delta = librosa.feature.delta(mel_db)
    delta2 = librosa.feature.delta(mel_db, order=2)

    mel_db_n = normalize(mel_db)
    delta_n = normalize(delta)
    delta2_n = normalize(delta2)

    stacked = np.stack([mel_db_n, delta_n, delta2_n])
    return stacked.astype(np.float32)


def make_features(audio: np.ndarray, sr: int) -> np.ndarray:
    """Notebook wrapper that returns the same feature format as training.

    Output:
    - `image`: np.ndarray float32 with shape (3, N_MELS, T)
    """
    _ = sr  # audio is already resampled to SR by `read_audio`

    mel_db = create_mel_spectrogram(audio)
    image = create_3_channel_tensor(mel_db)
    return image


## 6. Soundscape Windowing

This section reuses the **exact windowing strategy** from your local `generate_submission.py`:

- **5-second non-overlapping windows**
- `num_windows = total_samples // window_size` (drops remainder; no padding)
- `row_id = f"{filename}_{i*5}"`

This keeps the notebook compatible with the BirdCLEF submission row-id convention.


In [ ]:
def soundscape_num_windows(audio: np.ndarray) -> int:
    """Compute number of full 5s windows (local logic)."""
    window_size = int(SR * DURATION)
    total_samples = len(audio)
    return total_samples // window_size


def soundscape_window(audio: np.ndarray, i: int) -> np.ndarray:
    """Get i-th 5-second window (local logic)."""
    window_size = int(SR * DURATION)
    start = i * window_size
    end = start + window_size
    return audio[start:end]


def make_row_id(filename: str, window_index: int) -> str:
    """BirdCLEF row_id format used in local `generate_submission.py`."""
    return f"{filename}_{window_index * 5}"


def iter_soundscape_windows(audio: np.ndarray, filename: str) -> List[Tuple[str, np.ndarray]]:
    """Return list of (row_id, audio_clip) using local segmentation strategy.

    - non-overlapping 5s windows
    - drops remainder (no padding)
    """
    n = soundscape_num_windows(audio)
    out: List[Tuple[str, np.ndarray]] = []

    for i in range(n):
        clip = soundscape_window(audio, i)
        row_id = make_row_id(filename, i)
        out.append((row_id, clip))

    return out


## 7. Test-Time Augmentation (TTA)

This section reuses the **exact TTA logic** from your local inference pipeline (`generate_submission.py` / `tta_inference.py`).

Augmentations (no new methods):
- **time shift**: `np.roll(image, shift, axis=2)`
- **gain adjust**: multiply by `1.05` then `np.clip(..., -1, 1)`

Averaging strategy:
- run model on original + augmented inputs
- average probabilities via `torch.stack(predictions).mean(dim=0)`


In [ ]:
def time_shift(image: np.ndarray, shift: int = 10) -> np.ndarray:
    """Local TTA: shift along time axis (axis=2)."""
    return np.roll(image, shift, axis=2)


def gain_adjust(image: np.ndarray, gain: float = 1.05) -> np.ndarray:
    """Local TTA: gain scale + clip to [-1, 1]."""
    image = image * gain
    image = np.clip(image, -1, 1)
    return image


@torch.inference_mode()
def predict_tta(model: nn.Module, image: np.ndarray) -> torch.Tensor:
    """Local inference behavior: original + shift + gain, then mean."""
    predictions: List[torch.Tensor] = []

    # ORIGINAL
    x = torch.tensor(image, dtype=torch.float32).unsqueeze(0).to(TORCH_DEVICE)
    preds = torch.sigmoid(model(x))
    predictions.append(preds)

    # SHIFT
    shifted = time_shift(image, shift=10)
    x_shift = torch.tensor(shifted, dtype=torch.float32).unsqueeze(0).to(TORCH_DEVICE)
    preds_shift = torch.sigmoid(model(x_shift))
    predictions.append(preds_shift)

    # GAIN
    gained = gain_adjust(image, gain=1.05)
    x_gain = torch.tensor(gained, dtype=torch.float32).unsqueeze(0).to(TORCH_DEVICE)
    preds_gain = torch.sigmoid(model(x_gain))
    predictions.append(preds_gain)

    # AVERAGE
    final_preds = torch.stack(predictions).mean(dim=0)
    return final_preds.squeeze(0)


## 8. Ensemble Inference

Ensembling averages probabilities from multiple checkpoints/models.

Placeholders:
- use mean for baseline
- optionally add weights per model


In [ ]:
def ensemble_average(
    probs_1: np.ndarray,
    probs_2: np.ndarray,
    weights: Optional[Tuple[float, float]] = None,
) -> np.ndarray:
    """Local ensemble behavior.

    Local pipeline averages two model probability vectors:
        probs = (probs_1 + probs_2) / 2

    Optionally accepts weights, but defaults to equal weights to match local behavior.
    """
    p1 = np.asarray(probs_1, dtype=np.float32)
    p2 = np.asarray(probs_2, dtype=np.float32)

    if weights is None:
        return (p1 + p2) / 2.0

    w1, w2 = float(weights[0]), float(weights[1])
    denom = (w1 + w2) if (w1 + w2) != 0 else 1.0
    return (w1 * p1 + w2 * p2) / denom


## 9. Threshold Optimization

This notebook reuses the local project’s **pre-optimized thresholds** saved as `optimized_thresholds.csv`.

Behavior:
- load per-class thresholds into a `threshold_map`
- apply thresholds **per class** on predicted probabilities
- keep probability values unchanged (thresholding is used for final decisions / formatting)


In [ ]:
def load_threshold_map(path: str) -> Dict[str, float]:
    """Load `optimized_thresholds.csv` exactly like local `generate_submission.py`."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Threshold file not found: {path}")

    df = pd.read_csv(path)
    if "species" not in df.columns or "threshold" not in df.columns:
        raise ValueError("optimized_thresholds.csv must have columns: species, threshold")

    return dict(zip(df["species"].astype(str), df["threshold"].astype(float)))


def get_class_list(train_csv_path: str, threshold_map: Dict[str, float]) -> List[str]:
    """Match local label ordering (sorted primary labels)."""
    if os.path.exists(train_csv_path):
        train_df = pd.read_csv(train_csv_path)
        if "primary_label" in train_df.columns:
            return sorted(train_df["primary_label"].astype(str).unique().tolist())

    return sorted(list(threshold_map.keys()))


def apply_thresholds_per_class(
    probs: np.ndarray,
    class_list: List[str],
    threshold_map: Dict[str, float],
    default: float = 0.5,
) -> np.ndarray:
    """Apply per-class thresholds; keeps probabilities unchanged.

    Returns a boolean mask (n_classes,).
    """
    p = np.asarray(probs, dtype=np.float32).reshape(-1)
    if p.size != len(class_list):
        raise ValueError("Probability vector size must match class_list")

    thr = np.array([float(threshold_map.get(c, default)) for c in class_list], dtype=np.float32)
    return (p > thr)


def format_predictions(mask: np.ndarray, class_list: List[str]) -> str:
    """Format active classes as a space-separated string (common BirdCLEF schema)."""
    m = np.asarray(mask).reshape(-1)
    return " ".join([c for c, on in zip(class_list, m) if bool(on)])


## 10. Submission Generation

In [ ]:
def list_test_audio_files() -> List[Path]:
    """List `.ogg` files from competition test soundscapes directory."""
    if not os.path.exists(TEST_DIR):
        raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")

    return sorted(Path(TEST_DIR).glob("*.ogg"))


def build_taxonomy_to_model_idx(
    taxonomy: pd.DataFrame,
    label_to_idx: Dict[str, int],
) -> Dict[str, Optional[int]]:
    """Map taxonomy species IDs to model output indices (None = zero-shot)."""
    return {
        str(species_id): label_to_idx.get(str(species_id))
        for species_id in taxonomy["primary_label"].astype(str)
    }


def build_submission_df(
    sample: pd.DataFrame,
    per_row: Dict[str, np.ndarray],
    class_cols: List[str],
    species_to_model_idx: Dict[str, Optional[int]],
    threshold_map: Optional[Dict[str, float]] = None,
    class_list: Optional[List[str]] = None,
) -> pd.DataFrame:
    """Build submission aligned to sample_submission taxonomy species ID columns."""
    submission = sample.copy()

    if "row_id" not in submission.columns:
        raise ValueError("sample_submission.csv must contain a 'row_id' column")

    if class_cols:
        for i, row_id in enumerate(submission["row_id"].astype(str).tolist()):
            p = per_row.get(row_id)
            if p is None:
                continue
            for c in class_cols:
                model_idx = species_to_model_idx.get(str(c))
                if model_idx is not None:
                    submission.at[i, c] = float(p[model_idx])

    elif "predictions" in submission.columns:
        if class_list is None or threshold_map is None:
            raise ValueError("class_list and threshold_map required for predictions-column schema")
        pred_strings: List[str] = []
        for row_id in submission["row_id"].astype(str).tolist():
            p = per_row.get(row_id)
            if p is None:
                pred_strings.append("")
                continue
            mask = apply_thresholds_per_class(p, class_list, threshold_map, default=0.5)
            pred_strings.append(format_predictions(mask, class_list))
        submission["predictions"] = pred_strings

    else:
        raise ValueError("Unrecognized sample submission schema")

    return submission


# ---- Main execution (generate_submission.py equivalent) ----

sample = pd.read_csv(SAMPLE_SUBMISSION)
print("Sample shape:", sample.shape)

taxonomy = pd.read_csv(TAXONOMY_CSV)
train_df = pd.read_csv(TRAIN_CSV)
class_list = sorted(train_df["primary_label"].astype(str).unique().tolist())
label_to_idx = {label: idx for idx, label in enumerate(class_list)}
species_to_model_idx = build_taxonomy_to_model_idx(taxonomy, label_to_idx)

class_cols = [
    c for c in sample.columns
    if c not in ("row_id", "filename", "seconds", "site", "predictions")
]

print("Sample columns:", len(class_cols))
print("Model classes:", NUM_CLASSES)
print("Taxonomy rows:", len(taxonomy))

print("NUM_CLASSES:", NUM_CLASSES, "| class_list:", len(class_list))

threshold_map = load_threshold_map(THRESHOLDS_CSV)
print("Loaded thresholds:", len(threshold_map))

model_soundscape = load_model(SOUNDSCAPE_MODEL_PATH)
model_pseudo = load_model(PSEUDO_MODEL_PATH)
print("\n✅ Models loaded")

test_files = list_test_audio_files()
print("Found test audio files:", len(test_files))

per_row: Dict[str, np.ndarray] = {}
iterator = test_files if tqdm is None else tqdm(test_files)

for f in iterator:
    audio, _sr = read_audio(f, sr=SR)

    for row_id, clip in iter_soundscape_windows(audio, filename=f.name):
        mel = create_mel_spectrogram(clip)
        image = create_3_channel_tensor(mel)

        probs_1 = predict_tta(model_soundscape, image)
        probs_2 = predict_tta(model_pseudo, image)

        probs = (probs_1 + probs_2) / 2
        per_row[row_id] = probs.detach().cpu().numpy().reshape(-1)

print("Predicted rows:", len(per_row))

submission = build_submission_df(
    sample,
    per_row,
    class_cols,
    species_to_model_idx,
    threshold_map=threshold_map,
    class_list=class_list,
)

assert submission.shape == sample.shape, (
    f"Submission shape {submission.shape} != sample shape {sample.shape}"
)

submission.to_csv(
    "submission.csv",
    index=False,
)

print("submission.csv generated")
print("Submission shape:", submission.shape)
print(submission.head())

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 11. Final Summary

This notebook implements a complete **BirdCLEF inference + submission** pipeline aligned with real-world ecological audio classification.

### Ecological audio classification goals

BirdCLEF focuses on identifying bird species from challenging field recordings that may include:
- overlapping vocalizations
- background noise (wind, insects, rain)
- distance attenuation and recording artifacts
- long soundscapes where calls occur intermittently

### Key components in this pipeline

- **Soundscape adaptation**: long soundscapes are segmented into fixed 5-second windows (matching the competition’s row-id convention) so models trained on clip-like inputs can generalize to continuous recordings.
- **Pseudo labeling**: the project includes a pseudo-labeled model checkpoint trained with additional high-confidence labels, improving robustness to domain shift.
- **Ensemble inference**: predictions from the soundscape-adapted model and pseudo-labeled model are averaged to reduce variance and improve stability.
- **Test-Time Augmentation (TTA)**: the same lightweight TTAs used in the local pipeline (time shift + gain adjustment) are applied at inference and averaged.
- **Threshold optimization**: per-class thresholds loaded from `optimized_thresholds.csv` support better calibration across species with different detection difficulty and prevalence.

### Output

The notebook reads competition data from `/kaggle/input/competitions/birdclef-2026/` and writes:

```python
submission.to_csv("submission.csv", index=False)
```

Schema is determined by `sample_submission.csv`:
- **per-class probability columns** → probabilities are written directly
- **`predictions` column** → per-class thresholds format the final species string

### Future improvements

- **Sliding-window inference**: use overlap (`hop < window`) to reduce boundary misses.
- **Batching + caching**: batch window features per file and optionally cache spectrograms to speed up Kaggle inference.
- **Richer ensembling**: multi-fold checkpoints, weighted averaging, or logit ensembling.
- **Model upgrades**: transformer-based audio encoders (e.g., HTS-AT/BEATs) if runtime allows.
- **Better calibration**: temperature scaling or classwise calibration tuned on a held-out validation set.
